# Cynbit AI/ML Assessment
## AI-Powered Customer Review Intelligence System

Applicant: Bandarupalli Kusuma  
Role: AI/ML Engineer Intern

This notebook builds a complete proof of concept:
- Load a public labelled sentiment dataset.
- Inspect and clean review text.
- Train a TF-IDF + Logistic Regression classifier.
- Evaluate on held-out data.
- Detect likely complaint categories with transparent keyword rules.
- Recommend draft replies using templates.
- Create a Streamlit app, reports, README and requirements file.
- Package the project into a ZIP.

**Scope and honesty:** The dataset is public, not Cynbit's private data. This is an independent prototype proposal, not an existing Cynbit feature. Complaint detection uses keyword rules and suggested replies use templates; no LLM is called. Run the notebook to obtain real metrics; do not invent results.

## 0. Install dependencies
Run this cell first. Internet access is required.

In [ ]:
!pip -q install datasets scikit-learn pandas numpy matplotlib seaborn joblib streamlit

## 1. Imports and project folders

In [ ]:
import os, re, json, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = Path("/content/Cynbit-Customer-Review-Intelligence")
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"
DATA_DIR = PROJECT_DIR / "data"
for folder in [PROJECT_DIR, MODEL_DIR, REPORT_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

## 2. Download the public dataset
We use `jbeno/sentiment_merged` from Hugging Face. Check its dataset card and license before reuse.

In [ ]:
DATASET_NAME = "jbeno/sentiment_merged"
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print("Available splits:", list(raw_dataset.keys()))

## 3. Convert splits to Pandas and detect text/label columns

In [ ]:
frames = {name: split.to_pandas() for name, split in raw_dataset.items()}
for name, frame in frames.items():
    print(f"\\n{name}: {frame.shape}")
    print("Columns:", list(frame.columns))
    display(frame.head(3))

def find_column(columns, candidates, kind):
    lookup = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    raise ValueError(f"Could not find {kind} column. Available columns: {list(columns)}")

sample = next(iter(frames.values()))
TEXT_COL = find_column(sample.columns, ["sentence", "text", "review", "content", "review_text"], "text")
LABEL_COL = find_column(sample.columns, ["label", "sentiment", "target", "stars"], "label")
print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)

## 4. Standardize labels and create train/validation/test frames

In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    if any(w in s for w in ["negative", "neg", "unfavorable"]): return "negative"
    if any(w in s for w in ["neutral", "neu", "mixed", "objective"]): return "neutral"
    if any(w in s for w in ["positive", "pos", "favorable"]): return "positive"
    try:
        return int(float(s))
    except Exception:
        return s

def standardize(frame):
    result = frame[[TEXT_COL, LABEL_COL]].copy()
    result.columns = ["text", "raw_label"]
    result["text"] = result["text"].astype(str).str.strip()
    result["label_norm"] = result["raw_label"].apply(normalize_label)
    result = result.dropna(subset=["text", "label_norm"])
    return result[result["text"].str.len() > 0][["text", "label_norm"]].reset_index(drop=True)

std = {name: standardize(frame) for name, frame in frames.items()}
if "train" not in std:
    raise ValueError("Dataset has no train split.")

unique_labels = sorted(std["train"]["label_norm"].unique(), key=lambda x: str(x))
print("Normalized labels:", unique_labels)

if all(isinstance(x, str) and not x.isdigit() for x in unique_labels):
    label_map = {}
    for x in unique_labels:
        mapped = normalize_label(x)
        if mapped not in ["negative", "neutral", "positive"]:
            raise ValueError(f"Unrecognized label: {x}")
        label_map[x] = mapped
else:
    numeric = sorted(unique_labels, key=lambda x: float(x))
    if len(numeric) == 3:
        label_map = {numeric[0]: "negative", numeric[1]: "neutral", numeric[2]: "positive"}
    elif len(numeric) == 2:
        label_map = {numeric[0]: "negative", numeric[1]: "positive"}
    else:
        raise ValueError(f"Expected 2 or 3 classes, found: {numeric}")

def apply_map(frame):
    x = frame.copy()
    x["sentiment"] = x["label_norm"].map(label_map)
    return x.dropna(subset=["sentiment"])[["text", "sentiment"]].reset_index(drop=True)

std = {name: apply_map(frame) for name, frame in std.items()}
train_df = std["train"]
if "validation" in std:
    val_df = std["validation"]
elif "valid" in std:
    val_df = std["valid"]
else:
    train_df, val_df = train_test_split(train_df, test_size=0.10, random_state=RANDOM_STATE, stratify=train_df["sentiment"])

if "test" in std:
    test_df = std["test"]
else:
    train_df, test_df = train_test_split(train_df, test_size=0.15, random_state=RANDOM_STATE, stratify=train_df["sentiment"])

print("Label mapping:", label_map)
print("Train:", train_df.shape, "Validation:", val_df.shape, "Test:", test_df.shape)
print("Training classes:", sorted(train_df["sentiment"].unique()))

## 5. Data understanding and quality checks

In [ ]:
display(train_df.head(10))
print("Shape:", train_df.shape)
print("\nData types:\n", train_df.dtypes)
print("\nMissing values:\n", train_df.isnull().sum())
print("\nDuplicate rows:", train_df.duplicated().sum())
print("Duplicate texts:", train_df["text"].duplicated().sum())
print("\nSentiment counts:\n", train_df["sentiment"].value_counts())

plt.figure(figsize=(7,4))
sns.countplot(data=train_df, x="sentiment", order=["negative", "neutral", "positive"])
plt.title("Training Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of examples")
plt.tight_layout()
plt.show()

## 6. Text preprocessing
We preserve negation words such as `not`, because they can change sentiment.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for frame in [train_df, val_df, test_df]:
    frame["clean_text"] = frame["text"].apply(clean_text)

display(train_df[["text", "clean_text", "sentiment"]].head(10))

## 7. TF-IDF feature extraction

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=50000, ngram_range=(1,2),
    min_df=2, max_df=0.98, sublinear_tf=True,
    strip_accents="unicode"
)
X_train = vectorizer.fit_transform(train_df["clean_text"])
X_val = vectorizer.transform(val_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])
y_train, y_val, y_test = train_df["sentiment"], val_df["sentiment"], test_df["sentiment"]

print("Train matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)
print("Test matrix:", X_test.shape)
print("TF-IDF vocabulary size:", len(vectorizer.get_feature_names_out()))

## 8. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
model.fit(X_train, y_train)
val_pred = model.predict(X_val)
print("Validation accuracy:", round(accuracy_score(y_val, val_pred), 4))

## 9. Final test evaluation

In [ ]:
test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)
print("TEST ACCURACY:", round(test_accuracy, 4))
print("\nClassification report:")
print(classification_report(y_test, test_pred, zero_division=0))

labels_present = [x for x in ["negative", "neutral", "positive"] if x in set(y_test)]
cm = confusion_matrix(y_test, test_pred, labels=labels_present)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_present, yticklabels=labels_present)
plt.title("Confusion Matrix – Test Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 10. Complaint category detection
This baseline uses explainable keyword rules. It is not a separately trained topic model.

In [ ]:
ISSUE_KEYWORDS = {
    "delivery": ["delivery", "delivered", "shipping", "shipment", "late", "delay", "arrived"],
    "product_quality": ["quality", "broken", "damaged", "defective", "faulty", "poor", "durable"],
    "customer_support": ["support", "service", "staff", "representative", "response", "respond", "rude"],
    "pricing_payment": ["price", "cost", "expensive", "charge", "charged", "payment", "billing"],
    "app_or_website": ["app", "website", "login", "crash", "error", "slow", "loading"],
    "returns_or_refunds": ["return", "exchange", "refund", "replacement", "reimburse"]
}

def detect_issues(review):
    text = clean_text(review)
    found = []
    for category, words in ISSUE_KEYWORDS.items():
        if any(re.search(r"\b" + re.escape(word) + r"\b", text) for word in words):
            found.append(category)
    return found or ["general_feedback"]

print(detect_issues("The delivery was late and customer support did not respond."))

## 11. Suggested reply templates
Replies are drafts only and must be reviewed by a person before use.

In [ ]:
REPLIES = {
    "negative": "We are sorry for your experience. Thank you for bringing this to our attention. Our team will review the issue and work to improve.",
    "neutral": "Thank you for sharing your feedback. We appreciate your suggestions and will use them to improve our service.",
    "positive": "Thank you for your kind feedback! We are happy to hear about your experience and appreciate your support."
}
ISSUE_REPLIES = {
    "delivery": "We will review the delivery concern with our fulfilment team.",
    "product_quality": "We will share your product-quality feedback with the relevant team.",
    "customer_support": "We will review the support experience and work to improve response quality.",
    "pricing_payment": "We will review the pricing or payment concern with the relevant team.",
    "app_or_website": "We will share the technical issue with our product team.",
    "returns_or_refunds": "We will review the return or refund concern with the relevant team.",
    "general_feedback": "Your feedback will be shared with the relevant team."
}

def recommend_response(sentiment, issues):
    base = REPLIES.get(str(sentiment), REPLIES["neutral"])
    details = " ".join(ISSUE_REPLIES.get(issue, "") for issue in issues)
    return base + " " + details

## 12. Reusable prediction function

In [ ]:
def analyze_review(review):
    cleaned = clean_text(review)
    features = vectorizer.transform([cleaned])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    confidence = float(np.max(probabilities))
    probability_by_class = {str(label): float(p) for label, p in zip(model.classes_, probabilities)}
    issues = detect_issues(review)
    return {
        "review": review,
        "sentiment": str(prediction),
        "confidence": round(confidence, 4),
        "class_probabilities": probability_by_class,
        "detected_issues": issues,
        "suggested_response": recommend_response(prediction, issues)
    }

example = analyze_review("The delivery was very late and customer support did not help.")
print(json.dumps(example, indent=2))

## 13. Demo with sample reviews
These hand-written examples are for demonstration only; they are not training or test records.

In [ ]:
demo_reviews = [
    "The product is excellent and arrived quickly. I am very happy.",
    "The delivery was late and the support team did not respond.",
    "The product is okay. Nothing special, but it works.",
    "The app keeps crashing when I try to make a payment.",
    "Great quality and helpful customer service."
]
demo_results = [analyze_review(x) for x in demo_reviews]
demo_df = pd.DataFrame([{
    "review": r["review"],
    "predicted_sentiment": r["sentiment"],
    "confidence": r["confidence"],
    "detected_issues": ", ".join(r["detected_issues"]),
    "suggested_response": r["suggested_response"]
} for r in demo_results])
display(demo_df)

## 14. Save model artifacts and evaluation

In [ ]:
joblib.dump(model, MODEL_DIR / "sentiment_model.joblib")
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")

metadata = {
    "dataset": DATASET_NAME,
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "classes": [str(x) for x in model.classes_],
    "model": "LogisticRegression",
    "vectorizer": "TfidfVectorizer",
    "random_state": RANDOM_STATE,
    "test_accuracy": float(test_accuracy)
}
with open(MODEL_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

with open(REPORT_DIR / "evaluation_summary.json", "w") as f:
    json.dump({
        "test_accuracy": float(test_accuracy),
        "test_rows": int(len(test_df)),
        "classification_report": classification_report(y_test, test_pred, output_dict=True, zero_division=0)
    }, f, indent=2)

print("Model, vectorizer and evaluation saved.")

## 15. Generate Streamlit application

In [ ]:
app_source = r'''
import re
import joblib
import streamlit as st
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
MODEL_DIR = BASE_DIR / "models"
model = joblib.load(MODEL_DIR / "sentiment_model.joblib")
vectorizer = joblib.load(MODEL_DIR / "tfidf_vectorizer.joblib")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

ISSUES = {
    "delivery": ["delivery", "delivered", "shipping", "shipment", "late", "delay", "arrived"],
    "product_quality": ["quality", "broken", "damaged", "defective", "faulty", "poor"],
    "customer_support": ["support", "service", "staff", "response", "respond", "rude"],
    "pricing_payment": ["price", "cost", "expensive", "charge", "payment", "billing"],
    "app_or_website": ["app", "website", "login", "crash", "error", "slow", "loading"],
    "returns_or_refunds": ["return", "exchange", "refund", "replacement"]
}
REPLIES = {
    "negative": "We are sorry for your experience. Thank you for bringing this to our attention. Our team will review the issue and work to improve.",
    "neutral": "Thank you for sharing your feedback. We appreciate your suggestions and will use them to improve our service.",
    "positive": "Thank you for your kind feedback! We are happy to hear about your experience and appreciate your support."
}

def analyze(review):
    cleaned = clean_text(review)
    x = vectorizer.transform([cleaned])
    sentiment = str(model.predict(x)[0])
    probs = model.predict_proba(x)[0]
    found = []
    for category, words in ISSUES.items():
        if any(re.search(r"\b" + re.escape(word) + r"\b", cleaned) for word in words):
            found.append(category)
    if not found:
        found = ["general_feedback"]
    return sentiment, float(max(probs)), found, REPLIES.get(sentiment, REPLIES["neutral"])

st.set_page_config(page_title="Customer Review Intelligence", page_icon="💬", layout="wide")
st.title("AI-Powered Customer Review Intelligence")
st.caption("Cynbit AI/ML Engineer Intern assessment – independent proof of concept")
st.info("Public-dataset prototype. Suggested replies are drafts; review before sending.")
review = st.text_area("Enter a customer review", height=130)
if st.button("Analyze review", type="primary", disabled=not review.strip()):
    sentiment, confidence, issues, reply = analyze(review)
    c1, c2 = st.columns(2)
    c1.metric("Predicted sentiment", sentiment.title())
    c2.metric("Model confidence", f"{confidence:.1%}")
    st.subheader("Detected issue categories")
    st.write(", ".join(issues))
    st.subheader("Suggested response (review before sending)")
    st.write(reply)
st.caption("Not connected to customer accounts, review platforms or Cynbit production systems.")
'''
(PROJECT_DIR / "app.py").write_text(app_source, encoding="utf-8")
print("Created app.py")

## 16. Create reports, README and requirements

In [ ]:
readme = f"""# AI-Powered Customer Review Intelligence System

Independent AI/ML proof of concept for the Cynbit AI/ML Engineer Intern assessment.

## Features
- Sentiment classification using TF-IDF and Logistic Regression
- Test-set evaluation
- Keyword-based complaint category detection
- Template-based suggested responses
- Streamlit interface

## Dataset
Public Hugging Face dataset: `{DATASET_NAME}`. This is not Cynbit private data. Review the dataset card and license before redistribution or commercial use.

## Run locally
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Limitations
- Public dataset may not match real business review language.
- Complaint detection uses keyword rules.
- Replies are templates, not LLM-generated.
- Predictions may be wrong; human review is required.
- No production integration or automatic reply sending.

## Future improvements
- Evaluate on consented, domain-specific business reviews.
- Train a multi-label complaint classifier.
- Add multilingual support.
- Add human approval, monitoring and audit logging.
- Evaluate a guarded LLM response assistant.
"""
(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")

(PROJECT_DIR / "requirements.txt").write_text(
    "datasets\npandas\nnumpy\nscikit-learn\nmatplotlib\nseaborn\njoblib\nstreamlit\n",
    encoding="utf-8"
)

opportunity = """# AI Opportunity Report
## AI-Powered Customer Review Intelligence System

### Company and product context
This independent proposal explores a review-intelligence enhancement relevant to Cynbit's product area. It is not a claim that Cynbit has approved or deployed this feature.

### Business problem
Businesses receive customer feedback through digital channels. Manual review triage can be slow, and recurring issues such as delivery delays, product quality, billing and customer support may be difficult to prioritize.

### Identified AI opportunities
1. Sentiment classification to summarize positive, neutral and negative feedback.
2. Complaint category detection to group recurring issues.
3. Response recommendations to help support teams draft consistent replies.
4. Trend monitoring to identify changes in customer feedback.
5. Escalation support to flag repeated or high-priority complaints for human review.

### Selected opportunity
Build a Customer Review Intelligence prototype that classifies sentiment, identifies likely complaint topics and recommends a draft response.

### Expected business value
- Faster review triage.
- More consistent first-draft responses.
- Better visibility into recurring customer pain points.
- Improved prioritization for human support teams.

### Risks and controls
- Model errors: show confidence and require human review.
- Domain mismatch: validate with representative, consented business reviews.
- Privacy: minimize personal data and apply access/retention controls.
- Inaccurate replies: never auto-send; require approval.
- Bias and class imbalance: monitor per-class metrics.

### Success measures
Measure macro-F1, per-class recall, complaint-category precision, review triage time, human edit rate and customer-support satisfaction. Establish baselines before claiming business impact.

### Dataset limitation
The PoC uses a public Hugging Face sentiment dataset, not Cynbit private customer data. Results are not production performance claims.
"""
(REPORT_DIR / "AI_Opportunity_Report.md").write_text(opportunity, encoding="utf-8")

technical = f"""# Technical Approach Report
## AI-Powered Customer Review Intelligence System

### Architecture
Review text → cleaning → TF-IDF → Logistic Regression → sentiment prediction. In parallel, keyword rules identify likely complaint categories. Response templates create a suggested draft. Streamlit presents results for human review.

### Dataset and pipeline
Dataset: `{DATASET_NAME}` (public Hugging Face dataset). The notebook inspects columns and splits, standardizes text and labels, checks missing values and duplicates, and uses train/validation/test partitions where available. This is public proxy data, not company data.

### Preprocessing
Lowercase text, remove URLs and HTML tags, replace punctuation with spaces, normalize whitespace, and preserve negation terms.

### Feature engineering
TF-IDF with unigrams and bigrams, maximum vocabulary size 50,000, minimum document frequency 2, maximum document frequency 0.98 and sublinear term frequency.

### Model
Logistic Regression with up to 1,000 iterations and balanced class weights. A fixed random seed supports reproducibility.

### Evaluation
Accuracy, per-class precision, recall, F1-score and confusion matrix on held-out test data. Actual values are generated by running the notebook.

### Complaint detection
Keyword rules identify delivery, product quality, customer support, pricing/payment, app/website and returns/refunds. This baseline should be validated or replaced with a labelled multi-label classifier.

### Response generation and prompt architecture
This prototype uses templates, not an LLM. A future LLM component could receive review text, predicted sentiment and issue labels, with instructions to write a polite, factual draft, avoid promises or invented facts, protect personal data and require human approval.

### Deployment
The notebook creates `app.py`, `requirements.txt`, model artifacts and reports. Run `streamlit run app.py` after installing dependencies.

### Limitations
Public-dataset domain mismatch, keyword-based issue detection, template-based responses, English-language focus and no production integration. Human review is required.
"""
(REPORT_DIR / "Technical_Approach_Report.md").write_text(technical, encoding="utf-8")

sample = pd.DataFrame({
    "review": [
        "The product quality is excellent and delivery was quick.",
        "My order arrived late and support did not answer.",
        "The app is okay, but the payment page loads slowly."
    ],
    "note": ["synthetic_demo_example"] * 3
})
sample.to_csv(DATA_DIR / "sample_reviews.csv", index=False)
print("Documentation and sample file created.")

## 17. Package project files into a ZIP

In [ ]:
zip_path = Path("/content/Cynbit_Customer_Review_Intelligence_Project.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in PROJECT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(PROJECT_DIR.parent))

print("Project files:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PROJECT_DIR))
print("\nZIP:", zip_path)

## 18. Download the completed project ZIP to your phone

In [ ]:
from google.colab import files
files.download("/content/Cynbit_Customer_Review_Intelligence_Project.zip")

## 19. Optional deployment
The notebook creates `app.py`, but it does not publish a public web app automatically. Upload the generated project folder to GitHub and deploy it using Streamlit Community Cloud. The model files must be included in the repository for this simple deployment setup.

## 20. Final submission checklist
- [ ] Run every cell successfully.
- [ ] Inspect the actual test metrics and confusion matrix.
- [ ] Confirm model artifacts and both reports were generated.
- [ ] Download and inspect the ZIP.
- [ ] Upload the project to a public GitHub repository.
- [ ] Include setup instructions in README.
- [ ] Submit the GitHub URL and technical setup notes in the candidate portal.

Do not submit fabricated metrics or describe this prototype as production-ready.